In [ ]:
import sys
sys.path.append("../..")

import pandas as pd
import numpy as np
from python.functions.bridge import build_bridge_inputs, run_bridge

In [ ]:
target = 1_500_000

B = build_bridge_inputs()

BRIDGE = (B["model"], B["smearing"], B["hist_ln_C"], B["hist_ln_S"],
          B["fy_map"], B["net_add"], B["actual_back"])

# ARDL, NARDL, ARDL/NARDL ensemble and VECM forecasts
ardl_only  = run_bridge("../../data/outputs/forecasts/obr_scenario_forecasts.csv", "ardl_log", *BRIDGE)
nardl_only = run_bridge("../../data/outputs/forecasts/obr_scenario_forecasts.csv", "nardl_log", *BRIDGE)
ensemble   = run_bridge("../../data/outputs/forecasts/obr_scenario_forecasts.csv", "ensemble_log", *BRIDGE)
vecm       = run_bridge("../../data/outputs/forecasts/vecm_unconditional_forecast.csv", "vecm_log",
                        *BRIDGE, strip_space=True)

# Chronos-Bolt zero-shot forward path. Comes from a separate file in levels, so
# it needs the log transform and the column rename the others don't.
fc_chr = pd.read_csv("../../data/outputs/forecasts/chronos_forward.csv")
fc_chr["log_starts"] = np.log(fc_chr["starts"])
fc_chr = fc_chr.rename(columns={"Quarter": "period"})
chronos = run_bridge(fc_chr, "log_starts", *BRIDGE)

results = {"ARDL only (HEADLINE)": ardl_only, "NARDL only": nardl_only,
           "Ensemble": ensemble, "VECM (unconditional)": vecm,
           "Chronos (zero-shot univariate)": chronos}

for name, d in results.items():
    print(name)
    for fy, v in d.items():
        print(f"  {fy}: {v:,.0f}")
    cumulative = sum(d.values())
    print(f"  Cumulative: {cumulative:,.0f} ({100*cumulative/target:.1f}% of {target:,}, "
          f"shortfall {target-cumulative:,.0f})\n")

# NARDL vs ARDL comparison at net-additions level
ardl_total  = sum(ardl_only.values())
nardl_total = sum(nardl_only.values())
ensemble_total = sum(ensemble.values())

print(f"ARDL cumulative:      {ardl_total:,.0f}")
print(f"NARDL cumulative:     {nardl_total:,.0f}")
print(f"Diff (NARDL - ARDL):  {nardl_total - ardl_total:,.0f} homes "
      f"({100*(nardl_total - ardl_total)/489000:.1f}% of the ~489k shortfall)")
print(f"Ensemble midpoint check: {ensemble_total:,.0f} "
      f"(expected ~{(ardl_total+nardl_total)/2:,.0f} if it's a simple average)")

non-private new build is held flat at its recent average (it won't respond to the policy scenarios), and conversions/change-of-use/demolitions are likewise held at 2021-24 averages. 

In [ ]:
import matplotlib.pyplot as plt

fy_start = lambda s: int(str(s)[:4])  # "2022-23" -> 2022

# Actual history from LT120, plus the 2024-25 actual already in delivery
actual = B["lt120"]["Total net additional dwellings"].dropna()
actual.index = actual.index.map(fy_start)
if 2024 not in actual.index:
    actual.loc[2024] = ardl_only["2024-25"]
actual = actual.sort_index()

# Forecast path (2025-26 onward)
def fcast_series(d):
    s = pd.Series({fy_start(k): v for k, v in d.items() if fy_start(k) >= 2025}).sort_index()
    return pd.concat([actual.iloc[[-1]], s])   # prepend last actual to close the gap

# Prepend the last actual point so the forecast line connects without a gap
join_rdl  = fcast_series(ardl_only)
join_vecm = fcast_series(vecm)
join_chr  = fcast_series(chronos)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(actual.index, actual.values, color="#1f4e79", lw=2, label="Actual")
ax.plot(join_rdl.index,  join_rdl.values,  color="#c0392b", lw=2, ls="--",
        marker="o", label="ARDL/NARDL ensemble (OBR-conditioned)")
ax.plot(join_vecm.index, join_vecm.values, color="#e08e0b", lw=2, ls=":",
        marker="s", label="VECM (unconditional system)")
ax.plot(join_chr.index,  join_chr.values,  color="#2e7d32", lw=2, ls="-.",
        marker="^", label="Chronos-Bolt (zero-shot univariate)")
ax.set_ylabel("Net additional dwellings")
ax.set_xlabel("Financial year (start)")
ax.legend()
plt.tight_layout()
plt.show()

# Chronos

Chronos-Bolt is included in the forward projection above. The earlier
Chronos-T5 run was excluded: as a sampling/tokenising model its quarterly path
collapsed onto the token-bin lattice, taking only three distinct values across
the thirteen forecast quarters, so the annual totals were an artefact of the
bin grid rather than a forecast. Bolt is a direct quantile regressor with no
such lattice. The check below confirms the quarterly path is non-degenerate
before the annual figures are read.

In [ ]:
# Quarterly starts path behind the Chronos annual figures, with the 80% band.
# n_distinct == len(fc_chr) is the guard against the T5 lattice: if a future
# rerun of 03_chronos.py reverts to a sampling model, this assertion fires
# rather than the degenerate path silently reaching the annual totals.
n_distinct = fc_chr["starts"].nunique()
assert n_distinct == len(fc_chr), (
    f"quarterly path is degenerate -- only {n_distinct} distinct values across "
    f"{len(fc_chr)} quarters, which is the token-lattice failure mode"
)

print(f"Chronos quarterly starts ({n_distinct} distinct values / "
      f"{len(fc_chr)} quarters)\n")
print(fc_chr[["period", "lo", "starts", "hi"]]
      .rename(columns={"lo": "q10", "starts": "median", "hi": "q90"})
      .to_string(index=False, float_format=lambda v: f"{v:,.0f}"))

print("\nAnnual net additions")
for fy, v in chronos.items():
    print(f"  {fy}: {v:,.0f}")
chronos_total = sum(chronos.values())
print(f"  Cumulative: {chronos_total:,.0f} "
      f"({100*chronos_total/target:.1f}% of {target:,}, "
      f"shortfall {target-chronos_total:,.0f})")
print(f"\nvs ARDL headline: {chronos_total - ardl_total:+,.0f} homes")